# 00 — Quickstart: connect in sim, see a chain, place a paper trade

This notebook is your 10-minute tour of the bot's plumbing. Everything here runs
**offline in `sim` mode** — no Deribit keys, no network — and is **deterministic**
(seeded), so you get the same numbers every time.

You will:
1. **Connect** a sim market feed.
2. **Look at a `Chain`** (one expiry's option quotes) and its helpers.
3. **Place one paper trade** through the sim broker + portfolio.
4. **Read the journal** the grader reads.

> Golden rules to keep in mind: IV is a **decimal** (0.65 = 65%), time is in **years**,
> size is **signed contracts** (1 = 1 BTC), and option premiums are in **BTC** — convert
> to USD with the expiry's **forward** (`chain.forward`), not spot (`chain.index`).

In [1]:
import os, sys
# Notebooks run from the notebooks/ folder; hop up to the repo root so the
# `botkit`/`strategies` imports and the data/ + runs/ paths all resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

working dir: /Users/kalam/Podcast/iba-course-derivatives/f405-options-bot


## 1. Connect (sim mode is offline + deterministic)

`Config.load` reads `config.example.yaml`. `make_feed` hands back a `ReplayFeed` in
sim mode — it replays a captured chain (`data/deribit_snapshot.json`), picks the
expiry nearest ~7 days as 'the weekly', and evolves the BTC index along a seeded GBM
path, re-pricing every option each tick. `feed.chains()` yields one `Chain` per tick.

In [2]:
from botkit.config import Config
from botkit.feed import make_feed

cfg = Config.load("config.example.yaml")
cfg.mode = "sim"            # offline replay — no API keys, no network
print("mode:", cfg.mode, "| seed:", cfg.sim_seed, "| sim_days:", cfg.sim_days)

feed = make_feed(cfg)               # ReplayFeed in sim
chain = next(iter(feed.chains()))   # the first tick's front-weekly chain
print(f"chain @ {chain.ts}: index=${chain.index:,.0f}  forward=${chain.forward:,.0f}"
      f"  days_to_expiry={chain.days_to_expiry:.2f}  quotes={len(chain.quotes)}")

mode: sim | seed: 0 | sim_days: 7.0
chain @ 1782552419679: index=$60,246  forward=$60,287  days_to_expiry=5.94  quotes=44


## 2. Look at a `Chain`

A `Chain` is one expiry's snapshot. Handy helpers:
- `chain.atm('C'|'P')` — the at-the-money call/put (strike nearest the **forward**).
- `chain.nearest_delta(target, kind)` — e.g. the ~25-delta put.
- `chain.calls()` / `chain.puts()` — sorted by strike.

Each quote carries `mark` (in **BTC**), `mark_iv` (a **decimal**) and the Greeks.

In [3]:
import pandas as pd

atm_c = chain.atm("C")
atm_p = chain.atm("P")
print("ATM call:", atm_c.instrument_name, "| delta", round(atm_c.delta, 3), "| IV", round(atm_c.mark_iv, 3))
print("ATM put :", atm_p.instrument_name, "| delta", round(atm_p.delta, 3))
print("~25-delta put:", chain.nearest_delta(0.25, "P").instrument_name)

# A tidy table of the chain. Premiums are in BTC; convert to USD with the forward.
rows = [{
    "instrument": q.instrument_name, "kind": q.kind, "strike": q.strike,
    "mark_btc": round(q.mark, 5), "mark_usd": round((q.mark or 0) * chain.forward, 1),
    "iv": round(q.mark_iv, 3), "delta": round(q.delta, 3),
} for q in sorted(chain.quotes, key=lambda x: (x.kind, x.strike))]
df = pd.DataFrame(rows)
df.head(12)

ATM call: BTC-3JUL26-60000-C | delta 0.546 | IV 0.418
ATM put : BTC-3JUL26-60000-P | delta -0.454
~25-delta put: BTC-3JUL26-58000-P


,instrument,kind,strike,mark_btc,mark_usd,iv,delta
0,BTC-3JUL26-52000-C,C,52000.0,0.13860,8355.8,0.649,0.966
1,BTC-3JUL26-55000-C,C,55000.0,0.09068,5466.9,0.549,0.911
2,BTC-3JUL26-56000-C,C,56000.0,0.07536,4543.0,0.518,0.875
3,BTC-3JUL26-57000-C,C,57000.0,0.06063,3655.1,0.489,0.824
4,BTC-3JUL26-58000-C,C,58000.0,0.04691,2828.0,0.464,0.753
5,BTC-3JUL26-59000-C,C,59000.0,0.03436,2071.7,0.438,0.661
6,BTC-3JUL26-60000-C,C,60000.0,0.02370,1428.9,0.418,0.546
7,BTC-3JUL26-61000-C,C,61000.0,0.01498,903.0,0.398,0.418
8,BTC-3JUL26-62000-C,C,62000.0,0.00861,519.0,0.380,0.290
9,BTC-3JUL26-63000-C,C,63000.0,0.00463,278.9,0.372,0.183


## 3. Place ONE paper trade

The `SimBroker` fills a market order at the quote's mark crossed by half the spread
(a small deterministic slippage) and charges a Deribit-style taker fee. The
`Portfolio` is the accountant: it tracks signed positions + BTC cash and marks
everything to the chain. Let's **sell one ATM call** and watch the account change.

Selling a call leaves us **short a call** → negative net delta and **short vega**
(we now lose if vol rises). That is the seed of every short-vol strategy.

In [4]:
from botkit.broker import SimBroker
from botkit.portfolio import Portfolio
from botkit.journal import Journal
from botkit.types import Order

pf = Portfolio(cfg.start_equity_btc)   # start with 1 BTC of cash
broker = SimBroker()
jr = Journal("runs/quickstart")        # writes trades.csv / equity.csv / meta.json

# BEFORE: mark the (empty) book and log the opening equity row.
pf.mark(chain); before = pf.state(chain); jr.equity(before)
print(f"before: equity=${before.equity_usd:,.0f}  net_delta={before.greeks.delta:+.3f}")

# The single paper trade: sell one at-the-money call.
order = Order(atm_c.instrument_name, "sell", 1.0, label="quickstart_demo")
for f in broker.execute([order], chain):
    pf.apply(f)
    jr.trade(f, strategy="quickstart")
    print(f"filled: sell {f.amount} {f.instrument_name} @ {f.price:.4f} BTC (fee {f.fee:.5f} BTC)")

# AFTER: now short a call -> negative delta, short (negative) vega.
pf.mark(chain); after = pf.state(chain); jr.equity(after)
print(f"after : equity=${after.equity_usd:,.0f}  net_delta={after.greeks.delta:+.3f}"
      f"  net_vega=${after.greeks.vega:,.0f}")

jr.set_meta(strategy="quickstart", mode="sim", note="one demo trade")
jr.close()

before: equity=$60,246  net_delta=+0.000
filled: sell 1.0 BTC-3JUL26-60000-C @ 0.0232 BTC (fee 0.00030 BTC)
after : equity=$60,196  net_delta=-0.546  net_vega=$-3,047


## 4. Read the journal

Every run leaves three files in its `journal_dir`. **The CSV schemas are a grading
contract** — the score, the leaderboard and the autograder all read these columns by
name. Your strategy code never writes them; the runner does.

In [5]:
import pandas as pd
print("trades.csv")
display(pd.read_csv("runs/quickstart/trades.csv"))
print("equity.csv")
display(pd.read_csv("runs/quickstart/equity.csv"))

trades.csv


,ts,iso,instrument,side,amount,price_btc,price_usd,fee_btc,strategy,label
0,1782552419679,2026-06-27T09:26:59.679000+00:00,BTC-3JUL26-60000-C,sell,1.0,0.023202,1398.7584,0.0003,quickstart,quickstart_demo


equity.csv


,ts,iso,equity_usd,cash_btc,index,forward,net_delta,net_vega_usd,net_theta_usd,margin_util,realized_usd,unrealized_usd,liquidated
0,1782552419679,2026-06-27T09:26:59.679000+00:00,60245.610000,1.000000,60245.61,60287.22,0.000000,0.0000,0.0000,0.000000,0.0,0.00000,0
1,1782552419679,2026-06-27T09:26:59.679000+00:00,60196.427289,1.022902,60245.61,60287.22,-0.546244,-3047.4471,39167.9328,0.173964,0.0,-30.14361,0


---
**That's the whole loop:** feed → strategy → risk → broker → portfolio → journal.
The runner (`python -m botkit.runner`) just does this in a loop, one tick at a time.

Next: **`01_baselines_walkthrough.ipynb`** runs three reference strategies end-to-end
and shows *why* each one wins or loses.